# Thinking Models vs Instruct Models — Week 3

## Coding Task & General Knowledge Q&A

### Day 17–20: HumanEval Coding Benchmark
- 25 HumanEval coding problems
- Test all 4 models
- Record model responses and response time
- Check whether generated code runs correctly
- Score each response

### Day 21–23: General Knowledge Q&A
- 50 general knowledge questions
- Test all 4 models
- Record accuracy and clarity
- Review observable reasoning-style responses from Thinking Models
- Summarize observations

In [1]:
import os
import random
import time
import pandas as pd

from google.colab import userdata
from openai import OpenAI

# OpenAI
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY2")
client = OpenAI()

# DeepSeek-R1 - via Hugging Face router (Novita provider, pay-as-you-go
# HF credits; $0.70/M input, $2.50/M output tokens)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
deepseek_client = OpenAI(
    api_key=os.environ["HF_TOKEN"],
    base_url="https://router.huggingface.co/v1"
)

# Groq - GPT-OSS-20B
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
groq_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

print("API setup loaded successfully.")


# Retry helper: retry only transient errors (429/5xx), fail fast on
# fatal ones (401/402/403/404) and disable that model for the rest of the run.
# describe_api_error() tags *why* a call failed so credit/quota exhaustion
# (e.g. the HF-router 402s that killed DeepSeek earlier) is obvious at a glance
# instead of buried in a generic exception string.
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}
FATAL_STATUS_CODES = {400, 401, 402, 403, 404}
QUOTA_ERROR_PATTERN = "OUT OF CREDITS/QUOTA|RATE LIMITED|INVALID/MISSING API KEY"

disabled_models = set()


def describe_api_error(e):
    status_code = getattr(e, "status_code", None)
    message = str(e).lower()

    if status_code == 402 or "insufficient" in message or "credit" in message or "quota" in message:
        return "OUT OF CREDITS/QUOTA"
    if status_code == 429 or "rate limit" in message:
        return "RATE LIMITED"
    if status_code == 401 or "unauthorized" in message or "invalid api key" in message or "incorrect api key" in message:
        return "INVALID/MISSING API KEY"
    if status_code:
        return f"HTTP {status_code}"
    return type(e).__name__


def call_with_retry(model_name, api_call, max_attempts=3):
    if model_name in disabled_models:
        raise RuntimeError(f"[MODEL DISABLED] {model_name} was already disabled earlier this run after a non-retryable API error")

    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            return api_call()
        except Exception as e:
            last_error = e
            status_code = getattr(e, "status_code", None)
            tag = describe_api_error(e)

            if status_code in FATAL_STATUS_CODES:
                disabled_models.add(model_name)
                print(f"[DISABLED] {model_name}: {tag} - {e}")
                raise RuntimeError(f"[{tag}] {type(e).__name__}: {e}") from e

            if status_code not in RETRYABLE_STATUS_CODES or attempt == max_attempts:
                if attempt == max_attempts:
                    print(f"[GAVE UP after {max_attempts} attempts] {model_name}: {tag} - {e}")
                raise RuntimeError(f"[{tag}] {type(e).__name__}: {e}") from e

            print(f"[RETRY {attempt}/{max_attempts}] {model_name}: {tag} - retrying...")
            time.sleep((2 ** (attempt - 1)) + random.random())

    raise last_error

API setup loaded successfully.


In [2]:
def ask_gpt4o_mini(question):
    start_time = time.time()

    response = call_with_retry(
        "GPT-4o mini",
        lambda: client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": question}
            ]
        )
    )

    end_time = time.time()

    return response.choices[0].message.content, end_time - start_time


def ask_o3_mini(question):
    start_time = time.time()

    response = call_with_retry(
        "o3-mini",
        lambda: client.chat.completions.create(
            model="o3-mini",
            messages=[
                {"role": "user", "content": question}
            ]
        )
    )

    end_time = time.time()

    return response.choices[0].message.content, end_time - start_time


def ask_deepseek(question):
    start_time = time.time()

    response = call_with_retry(
        "DeepSeek-R1",
        lambda: deepseek_client.chat.completions.create(
            model="deepseek-ai/DeepSeek-R1:novita",
            messages=[
                {"role": "user", "content": question}
            ]
        )
    )

    end_time = time.time()

    return response.choices[0].message.content, end_time - start_time


def ask_gpt_oss(question):
    start_time = time.time()

    response = call_with_retry(
        "GPT-OSS-20B",
        lambda: groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "user", "content": question}
            ]
        )
    )

    end_time = time.time()

    return response.choices[0].message.content, end_time - start_time


MODELS = [
    ("GPT-4o mini", ask_gpt4o_mini),
    ("o3-mini", ask_o3_mini),
    ("DeepSeek-R1", ask_deepseek),
    ("GPT-OSS-20B", ask_gpt_oss),
]

model_type_map = {
    "DeepSeek-R1": "Thinking",
    "o3-mini": "Thinking",
    "GPT-4o mini": "Instruct",
    "GPT-OSS-20B": "Instruct"
}

print("All 4 model functions are ready.")

All 4 model functions are ready.


In [3]:
!pip -q install datasets

from datasets import load_dataset

humaneval = load_dataset("openai/openai_humaneval")

# Week 3: first 25 HumanEval problems only
humaneval_25 = humaneval["test"].select(range(25))

print("Total HumanEval problems selected:", len(humaneval_25))
print("First problem ID:", humaneval_25[0]["task_id"])
print("\nFirst problem prompt:\n")
print(humaneval_25[0]["prompt"])

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Total HumanEval problems selected: 25
First problem ID: HumanEval/0

First problem prompt:

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """



In [4]:
results = []

for problem in humaneval_25:

    task_id = problem["task_id"]
    prompt = problem["prompt"]

    for model_name, ask_fn in MODELS:
        try:
            response, response_time = ask_fn(prompt)

            results.append({
                "Task ID": task_id,
                "Prompt": prompt,
                "Model": model_name,
                "Response": response,
                "Response Time (sec)": round(response_time, 2)
            })

        except Exception as e:

            results.append({
                "Task ID": task_id,
                "Prompt": prompt,
                "Model": model_name,
                "Response": f"ERROR: {e}",
                "Response Time (sec)": None
            })


results_df = pd.DataFrame(results)

print("Total responses:", len(results_df))
print("Expected responses:", 25 * 4)

quota_issues = results_df[
    results_df["Response"].astype(str).str.contains(QUOTA_ERROR_PATTERN, case=False, na=False)
]

if len(quota_issues) > 0:
    print("\nAPI limit/credit related failures detected:")
    print(quota_issues.groupby("Model").size())
else:
    print("\nNo API limit/credit related failures detected.")

if disabled_models:
    print("\nModels disabled this run (non-retryable error):", disabled_models)

results_df.head(8)

[DISABLED] DeepSeek-R1: HTTP 404 - Error code: 404 - {'error': {'message': 'This model is unavailable for free. The paid version is available now - use this slug instead: deepseek/deepseek-r1', 'code': 404}, 'user_id': 'user_3Jx7vzh9tXd44hyjUJ69eEvMECK'}
Total responses: 100
Expected responses: 100

No API limit/credit related failures detected.

Models disabled this run (non-retryable error): {'DeepSeek-R1'}


,Task ID,Prompt,Model,Response,Response Time (sec)
0,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,GPT-4o mini,To implement the `has_close_elements` function...,7.43
1,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,o3-mini,One efficient approach is to first sort the li...,3.60
2,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,DeepSeek-R1,ERROR: [HTTP 404] NotFoundError: Error code: 4...,NaN
3,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,GPT-OSS-20B,"Here’s a compact, type‑annotated implementatio...",1.06
4,HumanEval/1,from typing import List\n\n\ndef separate_pare...,GPT-4o mini,To solve the task of separating groups of nest...,4.73
5,HumanEval/1,from typing import List\n\n\ndef separate_pare...,o3-mini,You can solve this problem by first removing a...,5.49
6,HumanEval/1,from typing import List\n\n\ndef separate_pare...,DeepSeek-R1,ERROR: [MODEL DISABLED] DeepSeek-R1 was alread...,NaN
7,HumanEval/1,from typing import List\n\n\ndef separate_pare...,GPT-OSS-20B,```python\nfrom typing import List\n\n\ndef se...,1.35


In [5]:
import re

def extract_code(response):
    if not isinstance(response, str):
        return ""

    # Python code block
    match = re.search(
        r"```python\s*(.*?)```",
        response,
        re.DOTALL | re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    # Any code block
    match = re.search(
        r"```\s*(.*?)```",
        response,
        re.DOTALL
    )

    if match:
        return match.group(1).strip()

    # If model did not use a code block
    return response.strip()


results_df["Generated Code"] = results_df["Response"].apply(extract_code)

print("Code extraction completed.")
print("Total rows:", len(results_df))

results_df[
    ["Task ID", "Model", "Generated Code"]
].head(4)

Code extraction completed.
Total rows: 100


,Task ID,Model,Generated Code
0,HumanEval/0,GPT-4o mini,from typing import List\n\ndef has_close_eleme...
1,HumanEval/0,o3-mini,One efficient approach is to first sort the li...
2,HumanEval/0,DeepSeek-R1,ERROR: [HTTP 404] NotFoundError: Error code: 4...
3,HumanEval/0,GPT-OSS-20B,from typing import List\nimport doctest\n\n\nd...


In [6]:
import subprocess
import sys
import tempfile


def run_humaneval_test(problem, generated_code, timeout=10):
    # Model-generated code must never run via exec() in this process - a bad
    # or infinite-looping generation would hang/crash the whole notebook.
    # Run it in a throwaway subprocess with a timeout instead.
    function_name = problem["entry_point"]
    test_code = problem["test"]

    full_script = (
        generated_code
        + "\n\n"
        + test_code
        + f"\n\ncandidate = {function_name}\ncheck(candidate)\n"
    )

    script_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".py", delete=False, encoding="utf-8"
        ) as f:
            f.write(full_script)
            script_path = f.name

        result = subprocess.run(
            [sys.executable, script_path],
            capture_output=True,
            text=True,
            timeout=timeout
        )

        if result.returncode == 0:
            return True, "All tests passed."

        error_lines = (result.stderr or result.stdout).strip().splitlines()
        return False, error_lines[-1] if error_lines else "Unknown error."

    except subprocess.TimeoutExpired:
        return False, f"Timeout: code did not finish within {timeout} seconds."

    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

    finally:
        if script_path:
            os.remove(script_path)


# Evaluate all 100 model responses
evaluation_results = []

for _, row in results_df.iterrows():

    problem = next(
        item for item in humaneval_25
        if item["task_id"] == row["Task ID"]
    )

    correct, test_result = run_humaneval_test(
        problem,
        row["Generated Code"]
    )

    evaluation_results.append({
        "Task ID": row["Task ID"],
        "Model": row["Model"],
        "Correct": correct,
        "Test Result": test_result
    })


evaluation_df = pd.DataFrame(evaluation_results)

print("Total evaluated responses:", len(evaluation_df))
print("Correct responses:", evaluation_df["Correct"].sum())
print("Incorrect responses:", (~evaluation_df["Correct"]).sum())

evaluation_df.head(8)

Total evaluated responses: 100
Correct responses: 52
Incorrect responses: 48


,Task ID,Model,Correct,Test Result
0,HumanEval/0,GPT-4o mini,True,All tests passed.
1,HumanEval/0,o3-mini,False,SyntaxError: unterminated string literal (dete...
2,HumanEval/0,DeepSeek-R1,False,SyntaxError: invalid syntax. Perhaps you forgo...
3,HumanEval/0,GPT-OSS-20B,True,All tests passed.
4,HumanEval/1,GPT-4o mini,True,All tests passed.
5,HumanEval/1,o3-mini,False,SyntaxError: invalid syntax
6,HumanEval/1,DeepSeek-R1,False,SyntaxError: invalid syntax. Perhaps you forgo...
7,HumanEval/1,GPT-OSS-20B,True,All tests passed.


In [7]:
humaneval_results_df = results_df.merge(
    evaluation_df,
    on=["Task ID", "Model"],
    how="left"
)

print("HumanEval final results table created.")
print("Total rows:", len(humaneval_results_df))
print(
    "Correct:",
    humaneval_results_df["Correct"].sum()
)
print(
    "Incorrect:",
    (~humaneval_results_df["Correct"]).sum()
)

humaneval_results_df[
    [
        "Task ID",
        "Model",
        "Response Time (sec)",
        "Generated Code",
        "Correct",
        "Test Result"
    ]
].head(8)

HumanEval final results table created.
Total rows: 100
Correct: 52
Incorrect: 48


,Task ID,Model,Response Time (sec),Generated Code,Correct,Test Result
0,HumanEval/0,GPT-4o mini,7.43,from typing import List\n\ndef has_close_eleme...,True,All tests passed.
1,HumanEval/0,o3-mini,3.60,One efficient approach is to first sort the li...,False,SyntaxError: unterminated string literal (dete...
2,HumanEval/0,DeepSeek-R1,NaN,ERROR: [HTTP 404] NotFoundError: Error code: 4...,False,SyntaxError: invalid syntax. Perhaps you forgo...
3,HumanEval/0,GPT-OSS-20B,1.06,from typing import List\nimport doctest\n\n\nd...,True,All tests passed.
4,HumanEval/1,GPT-4o mini,4.73,from typing import List\n\ndef separate_paren_...,True,All tests passed.
5,HumanEval/1,o3-mini,5.49,You can solve this problem by first removing a...,False,SyntaxError: invalid syntax
6,HumanEval/1,DeepSeek-R1,NaN,ERROR: [MODEL DISABLED] DeepSeek-R1 was alread...,False,SyntaxError: invalid syntax. Perhaps you forgo...
7,HumanEval/1,GPT-OSS-20B,1.35,from typing import List\n\n\ndef separate_pare...,True,All tests passed.


In [8]:
# A model call that failed outright (API error) must not count as "wrong code" -
# it has no code to judge. Detect those rows and exclude them from accuracy.
humaneval_results_df["API Error"] = (
    humaneval_results_df["Test Result"]
    .astype(str)
    .str.contains(
        r"APIStatusError|API Error|402|401|429|insufficient.*credit|credits.*depleted",
        case=False,
        na=False,
        regex=True
    )
)

humaneval_scorecard_rows = []

for model, data in humaneval_results_df.groupby("Model"):

    total = len(data)
    successful = data[~data["API Error"]]
    api_errors = int(data["API Error"].sum())

    correct_count = int((successful["Correct"] == True).sum())
    incorrect_count = int((successful["Correct"] == False).sum())

    if len(successful) > 0:
        accuracy = round(correct_count / len(successful) * 100, 2)
        avg_time = round(successful["Response Time (sec)"].mean(), 2)
        avg_quality = round(
            successful["Correct"].map({True: 5, False: 2}).mean(), 2
        )
    else:
        accuracy = avg_time = avg_quality = None

    humaneval_scorecard_rows.append({
        "Task": "HumanEval Coding",
        "Model": model,
        "Total Questions": total,
        "Successful Responses": len(successful),
        "Correct": correct_count,
        "Incorrect": incorrect_count,
        "API Errors": api_errors,
        "Accuracy (%)": accuracy,
        "Average Response Time (sec)": avg_time,
        "Average Quality (1-5)": avg_quality,
        "Model Type": model_type_map[model]
    })

humaneval_scorecard = pd.DataFrame(humaneval_scorecard_rows)

print("HumanEval scorecard created (API-error rows excluded from accuracy).")
display(humaneval_scorecard)

HumanEval scorecard created (API-error rows excluded from accuracy).


,Task,Model,Total Questions,Successful Responses,Correct,Incorrect,API Errors,Accuracy (%),Average Response Time (sec),Average Quality (1-5),Model Type
0,HumanEval Coding,DeepSeek-R1,25,25,0,25,0,0.0,NaN,2.00,Thinking
1,HumanEval Coding,GPT-4o mini,25,25,23,2,0,92.0,4.46,4.76,Instruct
2,HumanEval Coding,GPT-OSS-20B,25,25,25,0,0,100.0,0.88,5.00,Instruct
3,HumanEval Coding,o3-mini,25,25,4,21,0,16.0,3.06,2.48,Thinking


In [9]:
# Save complete HumanEval results
humaneval_results_df.to_csv(
    "Week3_HumanEval_Results.csv",
    index=False
)

# Save HumanEval scorecard
humaneval_scorecard.to_csv(
    "Week3_HumanEval_Scorecard_Final.csv",
    index=False
)

print("HumanEval files saved successfully.")
print("1. Week3_HumanEval_Results.csv")
print("2. Week3_HumanEval_Scorecard_Final.csv")

HumanEval files saved successfully.
1. Week3_HumanEval_Results.csv
2. Week3_HumanEval_Scorecard_Final.csv


In [10]:
from google.colab import files

files.download("Week3_HumanEval_Results.csv")
files.download("Week3_HumanEval_Scorecard_Final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# Week 3 — Task 3
# 50 General Knowledge Questions

general_knowledge_questions = [
    {"question_id": 1, "question": "What is the capital of Australia?"},
    {"question_id": 2, "question": "Which planet is known as the Red Planet?"},
    {"question_id": 3, "question": "Who wrote the play Romeo and Juliet?"},
    {"question_id": 4, "question": "Which is the largest ocean on Earth?"},
    {"question_id": 5, "question": "How many continents are there on Earth?"},
    {"question_id": 6, "question": "What is the chemical symbol for gold?"},
    {"question_id": 7, "question": "Which country is famous for the pyramids of Giza?"},
    {"question_id": 8, "question": "Which river is commonly considered the largest river by volume of water?"},
    {"question_id": 9, "question": "Who painted the Mona Lisa?"},
    {"question_id": 10, "question": "What is the largest animal on Earth?"},
    {"question_id": 11, "question": "Which gas do humans exhale during respiration?"},
    {"question_id": 12, "question": "What is the currency of Japan?"},
    {"question_id": 13, "question": "Which country is both a country and a continent?"},
    {"question_id": 14, "question": "Who was the first person to walk on the Moon?"},
    {"question_id": 15, "question": "At what temperature does water boil at sea level in Celsius?"},
    {"question_id": 16, "question": "Which organ pumps blood throughout the human body?"},
    {"question_id": 17, "question": "Which is the largest planet in our solar system?"},
    {"question_id": 18, "question": "What is the most widely spoken native language in the world?"},
    {"question_id": 19, "question": "What is the hardest naturally occurring mineral?"},
    {"question_id": 20, "question": "Which country is known as the world's largest democracy?"},
    {"question_id": 21, "question": "How many days are there in a leap year?"},
    {"question_id": 22, "question": "Which planet is closest to the Sun?"},
    {"question_id": 23, "question": "What is the capital city of Canada?"},
    {"question_id": 24, "question": "Which planet is famous for its prominent ring system?"},
    {"question_id": 25, "question": "Who developed the theory of relativity?"},
    {"question_id": 26, "question": "Which is the coldest continent on Earth?"},
    {"question_id": 27, "question": "What type of blood cells help the body fight infections?"},
    {"question_id": 28, "question": "What is the official language of Brazil?"},
    {"question_id": 29, "question": "Which instrument is used to measure temperature?"},
    {"question_id": 30, "question": "What is the process by which water vapor turns into liquid water?"},
    {"question_id": 31, "question": "Which country is famous for the Eiffel Tower?"},
    {"question_id": 32, "question": "What is the capital of South Korea?"},
    {"question_id": 33, "question": "Which vitamin is mainly produced in the skin through sunlight exposure?"},
    {"question_id": 34, "question": "Which organ is primarily responsible for filtering toxins from the blood?"},
    {"question_id": 35, "question": "Which ocean lies between Africa, Asia, and Australia?"},
    {"question_id": 36, "question": "Who discovered penicillin?"},
    {"question_id": 37, "question": "At what temperature does water freeze in Celsius?"},
    {"question_id": 38, "question": "Which is the largest country in the world by land area?"},
    {"question_id": 39, "question": "Which gas makes up the largest percentage of Earth's atmosphere?"},
    {"question_id": 40, "question": "Which planet is known for its beautiful rings?"},
    {"question_id": 41, "question": "What is the capital of Italy?"},
    {"question_id": 42, "question": "Which part of a plant usually performs photosynthesis?"},
    {"question_id": 43, "question": "How many sides does a hexagon have?"},
    {"question_id": 44, "question": "Which country has the largest population in South America?"},
    {"question_id": 45, "question": "What is the scientific study of earthquakes called?"},
    {"question_id": 46, "question": "Which planet is the largest in our solar system?"},
    {"question_id": 47, "question": "What is the capital of Germany?"},
    {"question_id": 48, "question": "What natural satellite orbits Earth?"},
    {"question_id": 49, "question": "What star is at the center of our solar system?"},
    {"question_id": 50, "question": "Which ocean surrounds the North Pole?"},
]

print("Total general knowledge questions:", len(general_knowledge_questions))
print("\nFirst question:", general_knowledge_questions[0])
print("Last question:", general_knowledge_questions[-1])

Total general knowledge questions: 50

First question: {'question_id': 1, 'question': 'What is the capital of Australia?'}
Last question: {'question_id': 50, 'question': 'Which ocean surrounds the North Pole?'}


In [12]:
# Run 50 GK Questions on All 4 Models

gk_results = []

for item in general_knowledge_questions:
    question_id = item["question_id"]
    question = item["question"]

    print(f"Processing Question {question_id}/50...")

    for model_name, ask_fn in MODELS:
        try:
            response, response_time = ask_fn(question)
            gk_results.append({
                "Question ID": question_id,
                "Question": question,
                "Model": model_name,
                "Response": response,
                "Response Time (sec)": round(response_time, 2),
                "Status": "Success"
            })
        except Exception as e:
            gk_results.append({
                "Question ID": question_id,
                "Question": question,
                "Model": model_name,
                "Response": "",
                "Response Time (sec)": None,
                "Status": f"Error: {type(e).__name__}: {e}"
            })

# Convert to DataFrame
gk_results_df = pd.DataFrame(gk_results)

print("\nGK test completed.")
print("Total responses:", len(gk_results_df))
print("Expected responses:", 50 * 4)

print("\nResponses by model:")
print(gk_results_df["Model"].value_counts())

print("\nStatus:")
print(gk_results_df["Status"].str.startswith("Success").value_counts())

quota_issues = gk_results_df[
    gk_results_df["Status"].astype(str).str.contains(QUOTA_ERROR_PATTERN, case=False, na=False)
]

if len(quota_issues) > 0:
    print("\nAPI limit/credit related failures detected:")
    print(quota_issues.groupby("Model").size())
else:
    print("\nNo API limit/credit related failures detected.")

if disabled_models:
    print("\nModels disabled this run (non-retryable error):", disabled_models)

Processing Question 1/50...
Processing Question 2/50...
Processing Question 3/50...
Processing Question 4/50...
Processing Question 5/50...
Processing Question 6/50...
Processing Question 7/50...
Processing Question 8/50...
Processing Question 9/50...
Processing Question 10/50...
Processing Question 11/50...
Processing Question 12/50...
Processing Question 13/50...
Processing Question 14/50...
Processing Question 15/50...
Processing Question 16/50...
Processing Question 17/50...
Processing Question 18/50...
Processing Question 19/50...
Processing Question 20/50...
Processing Question 21/50...
Processing Question 22/50...
Processing Question 23/50...
Processing Question 24/50...
Processing Question 25/50...
Processing Question 26/50...
Processing Question 27/50...
Processing Question 28/50...
Processing Question 29/50...
Processing Question 30/50...
Processing Question 31/50...
Processing Question 32/50...
Processing Question 33/50...
Processing Question 34/50...
Processing Question 35/

In [13]:
# Save raw General Knowledge results

gk_results_df.to_csv("Week3_GK_Results_Raw.csv", index=False)

print("GK raw results saved successfully.")
print("File: Week3_GK_Results_Raw.csv")
print("Total rows:", len(gk_results_df))

GK raw results saved successfully.
File: Week3_GK_Results_Raw.csv
Total rows: 200


In [14]:
# ============================================================
# WEEK 3 — GENERAL KNOWLEDGE COMPLETE EVALUATION
# No API calls are made in this cell.
# (gk_results_df was already saved to Week3_GK_Results_Raw.csv above)
# ============================================================

# ------------------------------------------------------------
# 2. Expected answers for all 50 questions
# ------------------------------------------------------------

expected_answers = {
    1: ["canberra"],
    2: ["mars"],
    3: ["william shakespeare", "shakespeare"],
    4: ["pacific ocean", "pacific"],
    5: ["7", "seven", "seven continents"],
    6: ["au", "gold"],
    7: ["egypt"],
    8: ["amazon river", "amazon"],
    9: ["leonardo da vinci", "leonardo", "da vinci"],
    10: ["blue whale"],
    11: ["carbon dioxide", "co2"],
    12: ["yen", "japanese yen"],
    13: ["australia"],
    14: ["neil armstrong", "armstrong"],
    15: ["100", "100 c", "100°c", "100 degrees celsius", "100 degrees"],
    16: ["heart"],
    17: ["jupiter"],
    18: ["mandarin chinese", "mandarin"],
    19: ["diamond"],
    20: ["india"],
    21: ["366", "three hundred sixty-six", "three hundred and sixty-six"],
    22: ["mercury"],
    23: ["ottawa"],
    24: ["saturn"],
    25: ["albert einstein", "einstein"],
    26: ["antarctica"],
    27: ["white blood cells", "white blood cell", "leukocytes", "leucocytes"],
    28: ["portuguese"],
    29: ["thermometer"],
    30: ["condensation"],
    31: ["france"],
    32: ["seoul"],
    33: ["vitamin d"],
    34: ["liver"],
    35: ["indian ocean"],
    36: ["alexander fleming", "fleming"],
    37: ["0", "0 c", "0°c", "0 degrees celsius", "zero degrees celsius"],
    38: ["russia"],
    39: ["nitrogen"],
    40: ["saturn"],
    41: ["rome"],
    42: ["leaves", "leaf"],
    43: ["6", "six"],
    44: ["brazil"],
    45: ["seismology"],
    46: ["jupiter"],
    47: ["berlin"],
    48: ["moon"],
    49: ["sun"],
    50: ["arctic ocean", "arctic"]
}


# ------------------------------------------------------------
# 3. Text normalization
# ------------------------------------------------------------

def normalize_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Make common symbols easier to compare
    text = text.replace("°", " ")
    text = text.replace("–", "-")
    text = text.replace("—", "-")

    # Remove extra punctuation
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ------------------------------------------------------------
# 4. Safer answer matching
# ------------------------------------------------------------

def answer_matches(response, accepted_answers):
    normalized_response = normalize_text(response)

    if not normalized_response:
        return False

    for answer in accepted_answers:
        normalized_answer = normalize_text(answer)

        # Numeric answers: require a separate token
        if normalized_answer.isdigit():
            if re.search(rf"\b{re.escape(normalized_answer)}\b",
                         normalized_response):
                return True

        # Text answers
        else:
            if normalized_answer in normalized_response:
                return True

    return False


# ------------------------------------------------------------
# 5. Evaluate every response
# ------------------------------------------------------------

evaluation_rows = []

for _, row in gk_results_df.iterrows():

    question_id = int(row["Question ID"])
    response = row["Response"]
    status = row["Status"]

    # API failure is NOT treated as an incorrect answer
    if not str(status).startswith("Success"):
        evaluation_status = "API Error"
        correct = None
    else:
        correct = answer_matches(
            response,
            expected_answers[question_id]
        )

        evaluation_status = "Correct" if correct else "Incorrect"

    evaluation_rows.append({
        "Question ID": question_id,
        "Question": row["Question"],
        "Model": row["Model"],
        "Response": response,
        "Response Time (sec)": row["Response Time (sec)"],
        "API Status": status,
        "Evaluation": evaluation_status,
        "Correct": correct
    })


gk_evaluation_df = pd.DataFrame(evaluation_rows)


# ------------------------------------------------------------
# 6. Automatic clarity score
#
# This is a simple project-level scoring rule:
# 5 = clear, direct answer with explanation
# 4 = clear answer
# 3 = answer present but somewhat verbose/unclear
# 2 = weak/incomplete response
# 1 = empty/unusable
# API errors are kept separately as 0 / API Error
# ------------------------------------------------------------

def clarity_score(row):

    if row["Evaluation"] == "API Error":
        return None

    response = row["Response"]

    if not isinstance(response, str) or not response.strip():
        return 1

    text = response.strip()
    word_count = len(text.split())

    # Very short but usable direct answers
    if word_count <= 8:
        return 4

    # Reasonably clear response
    if word_count <= 80:
        return 5

    # Longer response but still usable
    if word_count <= 180:
        return 4

    # Very long response
    return 3


gk_evaluation_df["Clarity Score (1-5)"] = (
    gk_evaluation_df.apply(clarity_score, axis=1)
)


# ------------------------------------------------------------
# 7. Model scorecard
# ------------------------------------------------------------

scorecard_rows = []

for model in gk_evaluation_df["Model"].unique():

    model_data = gk_evaluation_df[
        gk_evaluation_df["Model"] == model
    ]

    successful = model_data[
        model_data["Evaluation"] != "API Error"
    ]

    correct_count = successful["Correct"].sum()

    incorrect_count = (
        successful["Correct"] == False
    ).sum()

    api_error_count = (
        model_data["Evaluation"] == "API Error"
    ).sum()

    total = len(model_data)

    if len(successful) > 0:
        accuracy = round(
            correct_count / len(successful) * 100,
            2
        )
        avg_clarity = round(
            successful["Clarity Score (1-5)"].mean(),
            2
        )
        avg_response_time = round(
            successful["Response Time (sec)"].mean(),
            2
        )
    else:
        accuracy = None
        avg_clarity = None
        avg_response_time = None

    scorecard_rows.append({
        "Model": model,
        "Total Questions": total,
        "Successful Responses": len(successful),
        "Correct": int(correct_count),
        "Incorrect": int(incorrect_count),
        "API Errors": int(api_error_count),
        "Accuracy (%)": accuracy,
        "Average Clarity (1-5)": avg_clarity,
        "Average Response Time (sec)": avg_response_time
    })


gk_scorecard = pd.DataFrame(scorecard_rows)


# ------------------------------------------------------------
# 8. Thinking vs Instruct observations
# ------------------------------------------------------------

gk_evaluation_df["Model Type"] = (
    gk_evaluation_df["Model"].map(model_type_map)
)

model_type_summary = (
    gk_evaluation_df[
        gk_evaluation_df["Evaluation"] != "API Error"
    ]
    .groupby("Model Type")
    .agg(
        Successful_Responses=("Evaluation", "count"),
        Correct=("Correct", "sum"),
        Average_Clarity=("Clarity Score (1-5)", "mean"),
        Average_Response_Time=("Response Time (sec)", "mean")
    )
    .reset_index()
)

model_type_summary["Accuracy (%)"] = (
    model_type_summary["Correct"]
    / model_type_summary["Successful_Responses"]
    * 100
).round(2)

model_type_summary["Average_Clarity"] = (
    model_type_summary["Average_Clarity"].round(2)
)

model_type_summary["Average_Response_Time"] = (
    model_type_summary["Average_Response_Time"].round(2)
)


# ------------------------------------------------------------
# 9. Save all final files
# ------------------------------------------------------------

gk_evaluation_df.to_csv(
    "Week3_GK_Final_Evaluation.csv",
    index=False
)

gk_scorecard.to_csv(
    "Week3_GK_Scorecard_Final.csv",
    index=False
)

model_type_summary.to_csv(
    "Week3_GK_Thinking_vs_Instruct_Summary.csv",
    index=False
)


# ------------------------------------------------------------
# 10. Final display
# ------------------------------------------------------------

print("\n========================================")
print("WEEK 3 GK EVALUATION COMPLETED")
print("========================================")

print("\nTotal responses:", len(gk_evaluation_df))

print("\nEvaluation counts:")
print(gk_evaluation_df["Evaluation"].value_counts())

print("\nMODEL SCORECARD:")
display(gk_scorecard)

print("\nTHINKING VS INSTRUCT SUMMARY:")
display(model_type_summary)

print("\nFiles created:")
print("1. Week3_GK_Results_Raw.csv")
print("2. Week3_GK_Final_Evaluation.csv")
print("3. Week3_GK_Scorecard_Final.csv")
print("4. Week3_GK_Thinking_vs_Instruct_Summary.csv")


WEEK 3 GK EVALUATION COMPLETED

Total responses: 200

Evaluation counts:
Evaluation
Correct      150
API Error     50
Name: count, dtype: int64

MODEL SCORECARD:


,Model,Total Questions,Successful Responses,Correct,Incorrect,API Errors,Accuracy (%),Average Clarity (1-5),Average Response Time (sec)
0,GPT-4o mini,50,50,50,0,0,100.0,4.72,0.95
1,o3-mini,50,50,50,0,0,100.0,4.86,1.84
2,DeepSeek-R1,50,0,0,0,50,NaN,NaN,NaN
3,GPT-OSS-20B,50,50,50,0,0,100.0,4.66,0.35



THINKING VS INSTRUCT SUMMARY:


,Model Type,Successful_Responses,Correct,Average_Clarity,Average_Response_Time,Accuracy (%)
0,Instruct,100,100,4.69,0.65,100.0
1,Thinking,50,50,4.86,1.84,100.0



Files created:
1. Week3_GK_Results_Raw.csv
2. Week3_GK_Final_Evaluation.csv
3. Week3_GK_Scorecard_Final.csv
4. Week3_GK_Thinking_vs_Instruct_Summary.csv


In [15]:
from google.colab import files

files.download("Week3_GK_Results_Raw.csv")
files.download("Week3_GK_Final_Evaluation.csv")
files.download("Week3_GK_Scorecard_Final.csv")
files.download("Week3_GK_Thinking_vs_Instruct_Summary.csv")
print("All Week 3 GK files downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All Week 3 GK files downloaded.


In [16]:
# humaneval_scorecard already has the API-error-aware Task/Model Type columns
# baked in (built earlier, right after HumanEval evaluation) - just line up
# the GK scorecard to match it and combine.

gk_final = gk_scorecard.copy()

gk_final["Task"] = "General Knowledge"
gk_final["Model Type"] = gk_final["Model"].map(model_type_map)

gk_final["Average Quality (1-5)"] = gk_final["Average Clarity (1-5)"]

gk_final = gk_final[
    [
        "Task",
        "Model",
        "Total Questions",
        "Successful Responses",
        "Correct",
        "Incorrect",
        "API Errors",
        "Accuracy (%)",
        "Average Response Time (sec)",
        "Average Quality (1-5)",
        "Model Type"
    ]
]

final_scorecard = pd.concat(
    [humaneval_scorecard, gk_final],
    ignore_index=True
)

final_scorecard.to_csv(
    "Week3_Final_Scorecard.csv",
    index=False
)

humaneval_results_df.to_csv(
    "Week3_Final_Detailed_Results_HumanEval.csv",
    index=False
)

gk_evaluation_df.to_csv(
    "Week3_Final_Detailed_Results_GK.csv",
    index=False
)

print("Final Week 3 scorecard created.")
display(final_scorecard)

Final Week 3 scorecard created.


,Task,Model,Total Questions,Successful Responses,Correct,Incorrect,API Errors,Accuracy (%),Average Response Time (sec),Average Quality (1-5),Model Type
0,HumanEval Coding,DeepSeek-R1,25,25,0,25,0,0.0,NaN,2.00,Thinking
1,HumanEval Coding,GPT-4o mini,25,25,23,2,0,92.0,4.46,4.76,Instruct
2,HumanEval Coding,GPT-OSS-20B,25,25,25,0,0,100.0,0.88,5.00,Instruct
3,HumanEval Coding,o3-mini,25,25,4,21,0,16.0,3.06,2.48,Thinking
4,General Knowledge,GPT-4o mini,50,50,50,0,0,100.0,0.95,4.72,Instruct
5,General Knowledge,o3-mini,50,50,50,0,0,100.0,1.84,4.86,Thinking
6,General Knowledge,DeepSeek-R1,50,0,0,0,50,NaN,NaN,NaN,Thinking
7,General Knowledge,GPT-OSS-20B,50,50,50,0,0,100.0,0.35,4.66,Instruct


## Week 3 Summary

- 25 HumanEval coding problems were tested across all four models.
- Generated code was executed against the HumanEval test cases to check correctness.
- Response time was recorded for each model response.
- 50 general knowledge questions were tested across all four models.
- General knowledge responses were evaluated for accuracy and clarity.
- Observable response styles of the Thinking Models were reviewed and compared with the direct-answer style of the Instruct Models.
- DeepSeek-R1 General Knowledge results were recorded as API errors because the available Hugging Face inference credits were exhausted during testing.